In [ ]:
import urllib.request
import tarfile
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm

In [ ]:
data_dir = Path("data/flower_photos")
if not data_dir.exists():
    url = "http://download.tensorflow.org/example_images/flower_photos.tgz"
    archive_path, _ = urllib.request.urlretrieve(url)
    with tarfile.open(archive_path) as tar:
        tar.extractall(path="data")

In [ ]:
full_dataset = ImageFolder(str(data_dir))
class_names = full_dataset.classes
n_classes = len(class_names)
dataset_size = len(full_dataset)
print(n_classes, class_names, dataset_size)

In [ ]:
test_size = int(0.10 * dataset_size)
valid_size = int(0.15 * dataset_size)
train_size = dataset_size - test_size - valid_size

generator = torch.Generator().manual_seed(42)
test_set_raw, valid_set_raw, train_set_raw = random_split(
    full_dataset, [test_size, valid_size, train_size], generator=generator)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # Xception-style scaling to [-1, 1]
])
full_dataset.transform = preprocess

In [ ]:
batch_size = 32
train_set = DataLoader(train_set_raw, batch_size=batch_size, shuffle=True, num_workers=4)
valid_set = DataLoader(valid_set_raw, batch_size=batch_size, num_workers=4)
test_set = DataLoader(test_set_raw, batch_size=batch_size, num_workers=4)

In [ ]:
base_model = timm.create_model("xception", pretrained=True, num_classes=0)  # num_classes=0 -> pooled features, no head
feature_dim = base_model.num_features

model = nn.Sequential(
    base_model,
    nn.Linear(feature_dim, n_classes),
).to(device)

83697664/83683744 [==============================] - 0s 0us/step


In [ ]:
for param in base_model.parameters():
    param.requires_grad = False

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2, momentum=0.9)


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_correct, total_count = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            total_correct += (outputs.argmax(1) == labels).sum().item()
            total_count += images.size(0)
    return total_loss / total_count, total_correct / total_count


for epoch in range(5):
    train_loss, train_acc = run_epoch(model, train_set, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, valid_set, criterion)
    print(f"Epoch {epoch+1}/5 - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")

/usr/local/lib/python3.7/dist-packages/keras/optimizer_v2/gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


Epoch 1/5
86/86 [==============================] - 27s 159ms/step - loss: 2.1620 - accuracy: 0.7776 - val_loss: 1.4695 - val_accuracy: 0.8385
Epoch 2/5
86/86 [==============================] - 13s 145ms/step - loss: 0.5523 - accuracy: 0.9121 - val_loss: 0.9891 - val_accuracy: 0.8530
Epoch 3/5
86/86 [==============================] - 13s 146ms/step - loss: 0.2551 - accuracy: 0.9390 - val_loss: 0.8190 - val_accuracy: 0.8748
Epoch 4/5
86/86 [==============================] - 13s 147ms/step - loss: 0.1505 - accuracy: 0.9597 - val_loss: 0.8432 - val_accuracy: 0.8566
Epoch 5/5
86/86 [==============================] - 13s 147ms/step - loss: 0.1044 - accuracy: 0.9738 - val_loss: 0.8021 - val_accuracy: 0.8838


In [ ]:
for param in base_model.parameters():
    param.requires_grad = True

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

for epoch in range(10):
    train_loss, train_acc = run_epoch(model, train_set, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, valid_set, criterion)
    print(f"Epoch {epoch+1}/10 - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")

Epoch 1/10


/usr/local/lib/python3.7/dist-packages/keras/optimizer_v2/gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


86/86 [==============================] - 59s 613ms/step - loss: 0.4106 - accuracy: 0.8648 - val_loss: 0.6564 - val_accuracy: 0.8711
Epoch 2/10
86/86 [==============================] - 52s 604ms/step - loss: 0.0905 - accuracy: 0.9688 - val_loss: 0.4073 - val_accuracy: 0.8802
Epoch 3/10
86/86 [==============================] - 52s 605ms/step - loss: 0.0179 - accuracy: 0.9945 - val_loss: 0.3453 - val_accuracy: 0.9201
Epoch 4/10
86/86 [==============================] - 52s 610ms/step - loss: 0.0208 - accuracy: 0.9938 - val_loss: 0.4803 - val_accuracy: 0.8984
Epoch 5/10
86/86 [==============================] - 52s 605ms/step - loss: 0.0203 - accuracy: 0.9945 - val_loss: 0.3318 - val_accuracy: 0.9183
Epoch 6/10
86/86 [==============================] - 53s 610ms/step - loss: 0.0085 - accuracy: 0.9971 - val_loss: 0.3324 - val_accuracy: 0.9201
Epoch 7/10
86/86 [==============================] - 53s 618ms/step - loss: 0.0149 - accuracy: 0.9953 - val_loss: 0.3581 - val_accuracy: 0.9183
Epoch 8/10